In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import pandas as pd

In [4]:
df = pd.read_csv('/content/drive/MyDrive/Policing Project/Datasets/traffic_analysis.csv')

/tmp/ipykernel_10840/586767343.py:1: DtypeWarning: Columns (12,13,18,22,23) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('/content/drive/MyDrive/Policing Project/Datasets/traffic_analysis.csv')


In [6]:
from tqdm.auto import tqdm

In [ ]:
import json
import time
from typing import List, Dict

import pandas as pd
from openai import OpenAI

# pass your key directly here
api_key = ""

client = OpenAI(api_key=api_key)

MODEL = "gpt-5.4-nano"   # good fit for cheap, high-volume classification
BATCH_SIZE = 200         # tune this up/down based on token usage and rate limits
MAX_RETRIES = 5

SYSTEM_PROMPT = """
You are classifying traffic stop violation strings into one category.

Valid categories:
- License Issues
- Speeding
- Moving Violations
- Registration/Insurance
- Equipment
- Safety Violations
- Serious Offenses
- Pedestrian/Non-Driver
- Other

Definitions:
- License Issues: suspended, revoked, never issued, expired, unlicensed, invalid license, failure to obtain license.
- Speeding: any violation primarily about exceeding the speed limit.
- Moving Violations: non-speed driving behavior violations such as failure to signal, improper lane usage, following too close, stop sign/light violations, driving left of center, failure to reduce speed, improper passing, traffic control device violations.
- Registration/Insurance: registration expired, no valid registration, improper registration/title, plate issues, uninsured vehicle.
- Equipment: headlight, taillight, lighting, window tint, plate cover, and other vehicle equipment defects.
- Safety Violations: seat belt and similar occupant safety equipment violations.
- Serious Offenses: reckless driving, fleeing/eluding police, leaving the scene, DUI-related revoked/suspended offenses, failure to render aid, other clearly high-severity offenses.
- Pedestrian/Non-Driver: pedestrian violations, non-motorist obstruction, solicitation on roadway by pedestrian/non-driver.
- Other: anything else.

Rules:
1. Return one label per input violation.
2. Preserve input order exactly.
3. If multiple categories seem possible, choose the most specific offense type.
4. If the violation mentions speeding, classify as Speeding.
5. If the violation is about license status, classify as License Issues unless it is clearly DUI-related revoked/suspended driving, in which case classify as Serious Offenses.
6. If uncertain, return Other.
""".strip()

SCHEMA = {
    "type": "object",
    "properties": {
        "results": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "violation": {"type": "string"},
                    "grouped_vio": {
                        "type": "string",
                        "enum": [
                            "License Issues",
                            "Speeding",
                            "Moving Violations",
                            "Registration/Insurance",
                            "Equipment",
                            "Safety Violations",
                            "Serious Offenses",
                            "Pedestrian/Non-Driver",
                            "Other",
                        ],
                    },
                },
                "required": ["violation", "grouped_vio"],
                "additionalProperties": False,
            },
        }
    },
    "required": ["results"],
    "additionalProperties": False,
}


def classify_batch(batch: List[str]) -> Dict[str, str]:
    user_prompt = (
        "Classify each violation.\n"
        "Return JSON only.\n\n"
        "Violations:\n" +
        json.dumps(batch, ensure_ascii=False)
    )

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = client.responses.create(
                model=MODEL,
                store=False,
                input=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_prompt},
                ],
                text={
                    "format": {
                        "type": "json_schema",
                        "name": "violation_batch_labels",
                        "strict": True,
                        "schema": SCHEMA,
                    }
                },
            )

            data = json.loads(resp.output_text)
            rows = data["results"]

            # basic validation
            if len(rows) != len(batch):
                raise ValueError(
                    f"Expected {len(batch)} results, got {len(rows)}"
                )

            out = {}
            for expected, row in zip(batch, rows):
                if row["violation"] != expected:
                    raise ValueError(
                        f"Order/content mismatch. Expected {expected!r}, got {row['violation']!r}"
                    )
                out[expected] = row["grouped_vio"]

            return out

        except Exception:
            if attempt == MAX_RETRIES:
                raise
            time.sleep(min(2 ** attempt, 30))


# --- main logic ---

# leave NA as NA; only classify non-null unique values
unique_non_null = (
    pd.Series(df["violation"].dropna().unique())
    .astype(str)
    .tolist()
)

mapping = {}

total = len(unique_non_null)
num_batches = (total + BATCH_SIZE - 1) // BATCH_SIZE

with tqdm(total=total, desc="Classifying violations") as pbar:
    for start in range(0, total, BATCH_SIZE):
        batch = unique_non_null[start:start + BATCH_SIZE]
        result = classify_batch(batch)
        mapping.update(result)

        pbar.update(len(batch))

Classifying violations:   0%|          | 0/1109 [00:00<?, ?it/s]

ValueError: Expected 200 results, got 186

In [ ]:
df.head()